# Fake News Detection with Classical Machine Learning

This Google Colab project builds an end-to-end fake-news classification pipeline using TF-IDF text features, interpretable style indicators, and linear machine-learning models. It includes data-quality checks, leakage-aware feature design, model comparison, hold-out evaluation, error analysis, inference, and model export.


In [ ]:
# Environment setup

!pip -q install scikit-learn nltk==3.9.1 spacy==3.8.3
!python -m nltk.downloader stopwords punkt_tab wordnet omw-1.4
!python -m spacy download en_core_web_sm

In [ ]:
# Imports

import os
import re
import zipfile
import warnings
import urllib.request

import numpy as np
import pandas as pd
import joblib

from html import unescape
from collections import Counter

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Notebook settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

print("✅ All imports loaded successfully.")

In [ ]:
# Download and extract the dataset

ZIP_PATH    = "/content/fake_news.zip"
EXTRACT_DIR = "/content/fake_news"
DATA_URL    = "https://proai-datasets.s3.eu-west-3.amazonaws.com/fake_news.zip"

if not os.path.exists(ZIP_PATH):
    try:
        print("Downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print("✅ Download complete.")
    except Exception as e:
        raise RuntimeError(f"Download failed: {e}")
else:
    print("✅ Archive already present, skipping download.")

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)

true_df = pd.read_csv(f"{EXTRACT_DIR}/True.csv").assign(label=1)  # 21,417 rows
fake_df = pd.read_csv(f"{EXTRACT_DIR}/Fake.csv").assign(label=0)  # 23,481 rows
df_raw  = pd.concat([true_df, fake_df], ignore_index=True)

print(f"\nTrue.csv shape : {true_df.shape}")
print(f"Fake.csv shape : {fake_df.shape}")
print(f"Combined shape : {df_raw.shape}")
print(f"Columns        : {df_raw.columns.tolist()}")
display(df_raw.head(3))

## Data Processing & Feature Engineering

In [ ]:
# Initial EDA

df_raw.info()

print("\n=== Missing & Empty ===")
for col in ["title", "text"]:
    nulls  = df_raw[col].isna().sum()
    empty  = df_raw[col].astype(str).str.strip().eq("").sum()
    print(f"  {col}: {nulls} NaN, {empty} empty strings")

print(f"\n=== Duplicates ===")
print(f"  Full-row          : {df_raw.duplicated().sum()}")
print(f"  Title+text (cross): {df_raw.duplicated(subset=['title','text']).sum()}")

print("\n=== Class Distribution ===")
print(df_raw["label"].value_counts().rename({1:"Real", 0:"Fake"}))

print("\n=== Subject Distribution ===")
print(df_raw["subject"].value_counts())

In [ ]:
# Data cleaning

df = df_raw.copy()

# Drop duplicates
df = (df.drop_duplicates()
        .drop_duplicates(subset=["title","text"])
        .reset_index(drop=True))

# Normalize text
for col in ["title", "text"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Normalize subject
df["subject"] = (df["subject"].fillna("unknown")
                               .str.lower()
                               .str.replace(r"[\-_]", " ", regex=True)
                               .str.strip())

# Drop empty text
df = df[df["text"] != ""].reset_index(drop=True)

# Remove dates
df = df.drop(columns=["date"])

print(f"✅ Cleaned shape: {df.shape}")
display(df.head(3))

In [ ]:
# Text normalization

RAW_STOPWORDS = stopwords.words("english")
STOP_WORDS = set(w.lower() for w in RAW_STOPWORDS)

lemmatizer = WordNetLemmatizer()

def clean(text: str) -> str:
    """
    Basic cleaning:
    - unescape HTML entities
    - remove HTML tags and URLs
    - keep only letters and spaces
    - lowercase and collapse multiple spaces
    """
    text = unescape(str(text))
    text = re.sub(r"<.*?>|http\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\\s]", " ", text).lower()
    return re.sub(r"\\s+", " ", text).strip()

def lemmatize(text: str) -> str:
    """
    Lemmatize with NLTK WordNetLemmatizer and remove stopwords.
    """
    tokens = []
    for w in text.split():
        w_low = w.lower()
        if len(w_low) <= 2 or w_low in STOP_WORDS:
            continue
        tokens.append(lemmatizer.lemmatize(w_low))
    return " ".join(tokens)

def normalize(text: str) -> str:
    """
    Full normalization pipeline: clean + lemmatize.
    """
    return lemmatize(clean(text))

# Common buzzwords in clickbait / sensational headlines
SUSPICIOUS = {
    "shocking", "breaking", "bombshell", "urgent", "secret", "exposed",
    "viral", "alert", "truth", "click here", "mainstream media",
}

In [ ]:
# Feature Engineering

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build the feature matrix:
    - content: normalized title + text for TF-IDF
    - subject is intentionally excluded because it is source-specific and can leak the label
    - simple numeric style features from the raw title/text
    """
    X = df.copy()
    title = X["title"].astype(str)
    text  = X["text"].astype(str)

    # Main text feature for TF-IDF
    X["content"] = (title + " " + text).apply(normalize)

    # Length & word-count features
    X["title_len"]   = title.str.len()
    X["text_len"]    = text.str.len()
    X["title_words"] = title.str.split().str.len()
    X["text_words"]  = text.str.split().str.len()

    # Stylistic signals from the raw title
    X["exclamations"] = title.str.count("!")
    X["questions"]    = title.str.count(r"\?")
    X["upper_ratio"]  = title.apply(lambda s: sum(c.isupper() for c in s) / max(len(s), 1))

    # Sensational vocabulary
    X["suspicious_hits"] = title.str.lower().apply(
        lambda s: sum(term in s for term in SUSPICIOUS)
    )

    return X[[
        "content",
        "title_len", "text_len", "title_words", "text_words",
        "exclamations", "questions", "upper_ratio", "suspicious_hits",
    ]]

X = build_features(df)
y = df["label"]
display(X.head(3))

In [ ]:
# Train-Validation-Test Split

# 70% train, 15% validation, 15% test — stratified to preserve class balance
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=42
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print("Train label distribution:\n", y_train.value_counts(normalize=True))
print("Val label distribution:\n", y_val.value_counts(normalize=True))
print("Test label distribution:\n", y_test.value_counts(normalize=True))

In [ ]:
# Preprocessing pipeline

NUM_COLS = [
    "title_len", "text_len", "title_words", "text_words",
    "exclamations", "questions", "upper_ratio", "suspicious_hits",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                max_features=30000,
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True,
            ),
            "content",
        ),
        (
            "num",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler(with_mean=False)),
            ]),
            NUM_COLS,
        ),
    ],
    remainder="drop",
)

## Model Selection & Optimization

In [ ]:
# Baseline model comparison

MODELS = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "linear_svm": LinearSVC(
        class_weight="balanced",
        random_state=42,
    ),
}

results = []
pipelines = {}

for name, model in MODELS.items():
    pipe = Pipeline([
        ("pre", preprocessor),
        ("model", model),
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)

    # Accuracy and F1 are always available
    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds)

    # ROC AUC is available only for models that expose predict_proba.
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, proba)
    else:
        auc = None

    results.append({
        "model": name,
        "accuracy": round(acc, 4),
        "f1": round(f1, 4),
        "roc_auc": round(auc, 4) if auc is not None else None,
    })
    pipelines[name] = pipe

# Rank models by validation F1-score
results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

In [ ]:
# Use the best baseline model directly with default C=1.0
best_name = results_df.iloc[0]["model"]
best_pipeline = pipelines[best_name]
print(f"Best model selected: {best_name}")

## Final Evaluation on Test Set & Error Analysis

In [ ]:
# Evaluate the tuned model on the hold-out test set
preds = best_pipeline.predict(X_test)

if hasattr(best_pipeline, "predict_proba"):
    scores = best_pipeline.predict_proba(X_test)[:, 1]
elif hasattr(best_pipeline, "decision_function"):
    scores = best_pipeline.decision_function(X_test)
else:
    scores = None

auc = roc_auc_score(y_test, scores) if scores is not None else None

print(f"Accuracy : {accuracy_score(y_test, preds):.4f}")
print(f"F1-Score : {f1_score(y_test, preds):.4f}")
if auc: print(f"ROC-AUC  : {auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, preds, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))

In [ ]:
# Error analysis

err = X_test.reset_index(drop=True).copy()
err["true"] = y_test.reset_index(drop=True)
err["pred"] = preds

err["text"] = df.loc[y_test.index, "text"].reset_index(drop=True)
err = err[err["true"] != err["pred"]]

print(f"Total misclassified: {len(err)}")

display(err[["true", "pred", "text"]].head(10))

## Model Application

In [ ]:
# Inference helper

def predict_news(title, text, subject: str = "", date: str = ""):
    """
    Run inference on a single news article using the trained pipeline.
    Returns a dict with predicted label and (if available) a probability.
    """
    row = pd.DataFrame([{
        "title": title,
        "text": text,
        "subject": subject,
        "date": date,
    }])

    features = build_features(row)
    pred = best_pipeline.predict(features)[0]

    # Try to get a probability-like score, if supported
    proba = None
    if hasattr(best_pipeline, "predict_proba"):
        proba = best_pipeline.predict_proba(features)[0, 1]
    elif hasattr(best_pipeline, "decision_function"):
        # Map decision score to (0, 1) via sigmoid as an approximate confidence[web:67][web:114]
        score = best_pipeline.decision_function(features)[0]
        proba = 1 / (1 + np.exp(-score))

    return {
        "prediction": {1: "real", 0: "fake"}[pred],
        "real_probability": round(proba, 4) if proba is not None else None,
    }

# Quick test with a sensational-sounding headline
print(predict_news(
    title="BREAKING: Shocking secret exposed by mainstream media!",
    text="Click here to find out the truth they don't want you to know."
))

In [ ]:
# Save the full pipeline (preprocessor + model) for Chrome plugin integration
joblib.dump(best_pipeline, "/content/fake_news_pipeline.pkl")
print("Model saved to /content/fake_news_pipeline.pkl")

## Conclusions

### Result summary
This project builds a text-classification pipeline for distinguishing fake and real news. The predictive features combine TF-IDF representations of article titles and bodies with a small set of interpretable style indicators, including length, word counts, punctuation, uppercase ratio, and suspicious-keyword counts. Logistic Regression and Linear SVM are compared using validation F1, and the selected pipeline can be exported as `fake_news_pipeline.pkl`.

### Leakage-aware design
The dataset's `subject` categories are closely tied to the source collections and therefore to the target label. The published model intentionally excludes this field to reduce direct source leakage. Very high hold-out performance should still be interpreted cautiously because random row-level splits cannot fully measure generalization to unseen publishers, topics, or time periods.

### Limitations
- The dataset represents a limited set of publishers and historical topics.
- Text-only classification cannot detect manipulated images, audio, or video.
- A production evaluation should use source-separated and time-separated test sets.
- Model outputs are screening signals, not independent verification that an article is true or false.

### Next step for deployment
Load `fake_news_pipeline.pkl` in a small Python service such as FastAPI and connect it to a client application for real-time predictions, while displaying confidence and appropriate human-review guidance.
